In [1]:
import sys
from pathlib import Path
import torch
from transformers import TrainingArguments

ROOT = Path.cwd().resolve().parent
sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
paths = get_paths(ROOT)


from teacher_finetune import(
    build_teacher_tokenizer, 
    build_teacher_model, 
    TeacherModelConfig, 
    load_chunks_dataset, 
    build_collator, 
    WeightedBCETrainer,
    bf16_supported,
    freeze_bert_layers
)

paths

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ProjectPaths(root=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526'), src=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/src'), notebooks=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/notebooks'), data_processed=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed'), data_raw=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/raw'), checkpoints=WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/checkpoints'))

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"BF16 Supported: {bf16_supported()}")
torch.backends.cuda.matmul.allow_tf32 = True 
torch.backends.cudnn.allow_tf32 = True

MODEL_NAME = "bert-large-uncased"
cfg = TeacherModelConfig(
    model_name=MODEL_NAME,
    hidden_dropout_prob=0.1,
    gradient_checkpointing=False  
)

tokenizer = build_teacher_tokenizer(MODEL_NAME)
model = build_teacher_model(cfg)

freeze_bert_layers(model, freeze_embeddings=True, freeze_layers=18)
model.to(device)

Device: cuda
BF16 Supported: True


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-large-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Embeddings FROZEN.
First 18/24 layers of encoder have been frozen.
Trainable parameters: 76.6M / 335.1M (22.9%)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1

In [3]:


train_chunks_path = paths.data_processed / "train_80k_chunks.parquet"
val_chunks_path   = paths.data_processed / "val_80k_chunks.parquet"
#test_chunks_path  = paths.data_processed / "test_80k_chunks.parquet"
train_counts_path = paths.data_processed / "train_120k_chunk_counts.parquet"

assert train_chunks_path.exists()
assert val_chunks_path.exists()
#assert test_chunks_path.exists()
assert train_counts_path.exists()

(train_chunks_path, val_chunks_path, train_counts_path)


(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/val_80k_chunks.parquet'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/train_120k_chunk_counts.parquet'))

In [4]:

train_ds = load_chunks_dataset(
    chunks_parquet=train_chunks_path,
    chunk_counts_parquet=train_counts_path,
    add_sample_weight=True,
)
val_ds = load_chunks_dataset(
    val_chunks_path, 
    train_counts_path, 
    add_sample_weight=True
)

print(f"Train size: {len(train_ds)}")
print(f"Val size: {len(val_ds)}")


Add sample_weight: 100%|██████████| 26663/26663 [00:01<00:00, 24207.87 examples/s]

Train size: 212943
Val size: 26663


In [5]:
collator = build_collator(tokenizer)
BATCH_SIZE = 16
ACCUMULATION = 2

output_dir = paths.checkpoints / "teacher_bert_large"

training_args = TrainingArguments(
    output_dir=str(output_dir),
    overwrite_output_dir=True,
    num_train_epochs=2,              
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=ACCUMULATION,
    bf16=True,                       
    tf32=True,                       
    optim="adamw_bnb_8bit",         
    
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    logging_steps=50,                
    eval_strategy="steps",
    eval_steps=500,                  
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    
    dataloader_num_workers=4,
    group_by_length=True,            
    
    remove_unused_columns=False,     
    report_to="none"                 
)

In [6]:
trainer = WeightedBCETrainer(
    model=model,
    args = training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
)

In [7]:
print(f"Effective Batch Size: {BATCH_SIZE * ACCUMULATION}")
trainer.train()

final_path = paths.checkpoints / "teacher_bert_large"
trainer.save_model(str(final_path))
tokenizer.save_pretrained(str(final_path))
print(f"Model saved into: {final_path}")

Effective Batch Size: 32


Step,Training Loss,Validation Loss
500,0.236900,0.653171
1000,0.228600,0.575524
1500,0.232500,0.619055
2000,0.223800,0.556490
2500,0.227800,0.566640
3000,0.223300,0.554534


KeyboardInterrupt: 